# 06 - Autoencoder para Deteccion de Anomalias

## Objetivo
Entrenar un autoencoder SOLO con datos normales para detectar anomalias.

## Concepto Clave
El autoencoder aprende a reconstruir patrones normales.
Cuando un dato anomalo entra, la reconstruccion falla y el error es alto.
Error alto = Anomalia detectada.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, roc_auc_score, precision_recall_curve

from src.data.preprocessor import FiberDataPreprocessor
from src.models.autoencoder import FiberAutoencoder
from src.utils.config_loader import load_config, get_path
from src.utils.visualization import apply_style

apply_style()
config = load_config()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# Preparar datos
preprocessor = FiberDataPreprocessor()
train_df, val_df, test_df = preprocessor.process_pipeline()

feature_cols = ['optical_power_dbm', 'attenuation_db_km', 'ber', 'osnr_db',
                'chromatic_dispersion', 'temperature', 'humidity', 'log_ber']
feature_cols = [c for c in feature_cols if c in train_df.columns]

# IMPORTANTE: Entrenar SOLO con datos normales
train_normal = train_df[train_df['fault_type'] == 'normal'][feature_cols].values
val_all = val_df[feature_cols].values
val_labels = val_df['is_fault'].values
test_all = test_df[feature_cols].values
test_labels = test_df['is_fault'].values

print(f'Train (solo normal): {train_normal.shape}')
print(f'Test (todo): {test_all.shape}, Anomalias: {test_labels.sum()}')

In [ ]:
# Crear modelo
ae_cfg = config['models']['autoencoder']
model = FiberAutoencoder(
    n_features=len(feature_cols),
    encoder_dims=ae_cfg['encoder_dims'],
    dropout=ae_cfg['dropout'],
).to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=ae_cfg['learning_rate'])

# Convertir a tensores
train_tensor = torch.FloatTensor(train_normal).to(device)
train_dataset = torch.utils.data.TensorDataset(train_tensor, train_tensor)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=ae_cfg['batch_size'], shuffle=True)

In [ ]:
# Training Loop
epochs = ae_cfg['epochs']
train_losses = []

for epoch in range(epochs):
    model.train()
    epoch_loss = 0
    for batch_x, _ in train_loader:
        optimizer.zero_grad()
        reconstructed = model(batch_x)
        loss = criterion(reconstructed, batch_x)  # Comparar reconstruccion con input
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    
    epoch_loss /= len(train_loader)
    train_losses.append(epoch_loss)
    
    if (epoch + 1) % 20 == 0:
        print(f'Epoch {epoch+1}/{epochs} | Loss: {epoch_loss:.6f}')

print(f'\nLoss final: {train_losses[-1]:.6f}')

In [ ]:
# Curva de entrenamiento
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(train_losses, color='steelblue', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('Autoencoder - Curva de Entrenamiento')
plt.tight_layout()
plt.show()

In [ ]:
# Calcular umbral y detectar anomalias
model.eval()
model.cpu()

normal_tensor = torch.FloatTensor(train_normal[:5000])  # Submuestra para umbral
threshold = model.compute_threshold(normal_tensor, percentile=ae_cfg['anomaly_percentile'])

test_tensor = torch.FloatTensor(test_all)
is_anomaly, errors = model.detect_anomalies(test_tensor)

print(f'Umbral: {threshold:.6f}')
print(f'Anomalias detectadas: {is_anomaly.sum()} de {len(is_anomaly)}')
print(f'Anomalias reales: {test_labels.sum()}')

In [ ]:
# Distribucion de errores
fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(errors[test_labels == 0], bins=100, alpha=0.6, label='Normal', color='#2ecc71', density=True)
ax.hist(errors[test_labels == 1], bins=100, alpha=0.6, label='Anomalia', color='#e74c3c', density=True)
ax.axvline(x=threshold, color='black', linestyle='--', label=f'Umbral ({threshold:.4f})')
ax.set_xlabel('Error de Reconstruccion (MSE)')
ax.set_ylabel('Densidad')
ax.set_title('Distribucion de Errores de Reconstruccion')
ax.legend()
plt.tight_layout()
plt.show()

# Metricas
print('\nClassification Report:')
print(classification_report(test_labels, is_anomaly.astype(int), target_names=['Normal', 'Anomalia']))

# Guardar modelo
torch.save(model.state_dict(), str(get_path('models') / 'autoencoder_best.pt'))